* **Block 1:** Create Google Colab Notebook + Install Libraries*

In [1]:
!pip install PyPDF2 python-docx nltk spacy scikit-learn pandas numpy streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.1 MB/s eta 0:00:00


**Block 2: Import Libraries**

In [2]:
# Data Handling
import pandas as pd
import numpy as np

# Resume Reading
import PyPDF2
import docx

# NLP
import nltk
import spacy

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Save Model
import pickle

# File Handling
import os

print("Libraries Imported Successfully ✅")

Libraries Imported Successfully ✅


**Block 3: Download NLP Resources**

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

print("NLTK Setup Complete ✅")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...


NLTK Setup Complete ✅


[nltk_data]   Unzipping corpora/stopwords.zip.


**Block 4: Load English NLP Model**

In [4]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("SpaCy Model Loaded Successfully ✅")

SpaCy Model Loaded Successfully ✅


**Block 5: Test Environment**

In [6]:
text = """
I am a Python Developer.
I have experience in Machine Learning,
TensorFlow and NLP.
"""

doc = nlp(text)

for token in doc:
    print(token.text, token.pos_)


 SPACE
I PRON
am AUX
a DET
Python PROPN
Developer PROPN
. PUNCT

 SPACE
I PRON
have VERB
experience NOUN
in ADP
Machine PROPN
Learning PROPN
, PUNCT

 SPACE
TensorFlow PROPN
and CCONJ
NLP PROPN
. PUNCT

 SPACE


**Block 6: Upload Resume PDF and Extract Text**
Cell 6.1: Upload Resume File

In [11]:
from google.colab import files

uploaded = files.upload()

resume_file = list(uploaded.keys())[0]

print("Uploaded File:", resume_file)

Saving Sohail Akhtar CV.pdf to Sohail Akhtar CV.pdf
Uploaded File: Sohail Akhtar CV.pdf


**Cell 6.2: Extract Text From PDF**

In [12]:
def extract_pdf_text(pdf_file):

    pdf_reader = PyPDF2.PdfReader(pdf_file)

    text = ""

    for page in pdf_reader.pages:
        text += page.extract_text()

    return text

**Cell 6.3: Read Uploaded Resume**

In [13]:
with open(resume_file, "rb") as file:

    resume_text = extract_pdf_text(file)


print(resume_text[:1000])

Experience
Education
Skills SummaryLanguage
Expertise
80%
50%WordPress Developer
I've been working with WordPress for a
year now, focusing on making websites
look great and work smoothly. I love
solving problems and making sure
everything runs just right. Let's team up
and create something awesome for
your online presence!About Me
akhtersohail141@gmail.com+923003882593Upwrok
2021 - 2022
Bachelors of Computer Science
2018 - 2022Faisal colony Sheikhupura
PakistanI have good experience with Upwork. I have worked on it for one
year. I completed so many projects and get good experience in
this field.
English
Urdu (basic)
Punjabi (basic)
WordPress Developer
Theme customization
Website SEO
Content Writer
HTML and CSS
PHP and MYSQLBlogging Website
2023 - 2024
I created my blogging website the niche is technology. After
creating this blog I get so much knowledge in Wordpress. I
know how to customize themes, website design, website
SEO, and content writing. Website analysis and
technicalissues. 

**Block 7: Clean Resume Text**

Machine Learning models don't understand raw text.

We convert:

Before:

Python Developer!!!
Experience: 2 Years.
TensorFlow & NLP

After:

python developer experience 2 years tensorflow nlp

**Cell 7.1: Import Cleaning Tools**

In [14]:
import re

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

**Cell 7.2: Text Cleaning Function**

In [15]:
def clean_text(text):

    # lowercase
    text = text.lower()

    # remove special characters
    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        '',
        text
    )

    # remove extra spaces
    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    # remove stopwords
    words = text.split()

    words = [
        word for word in words
        if word not in stop_words
    ]

    return " ".join(words)

**Cell 7.3: Apply Cleaning**

In [16]:
clean_resume = clean_text(resume_text)

print(clean_resume[:1000])

experience education skills summarylanguage expertise 80 50wordpress developer ive working wordpress year focusing making websites look great work smoothly love solving problems making sure everything runs right lets team create something awesome online presenceabout akhtersohail141gmailcom923003882593upwrok 2021 2022 bachelors computer science 2018 2022faisal colony sheikhupura pakistani good experience upwork worked one year completed many projects get good experience field english urdu basic punjabi basic wordpress developer theme customization website seo content writer html css php mysqlblogging website 2023 2024 created blogging website niche technology creating blog get much knowledge wordpress know customize themes website design website seo content writing website analysis technicalissues virtual university pakistan wordpress developer website seosohail akhtar


**Block 8: Create Skill Database**

Now we create our AI skill detector.

In [17]:
skills_database = [

"python",
"machine learning",
"deep learning",
"tensorflow",
"pytorch",
"scikit learn",
"nlp",
"computer vision",
"opencv",
"sql",
"mongodb",
"aws",
"docker",
"streamlit",
"git",
"github",
"java",
"javascript",
"react",
"linux"

]

print("Skills Loaded:", len(skills_database))

Skills Loaded: 20


**Cell 8.2: Extract Skills From Resume**

In [18]:
def extract_skills(text):

    found_skills = []

    for skill in skills_database:

        if skill in text:
            found_skills.append(skill)

    return found_skills

**Cell 8.3: Test Skill Extraction**

In [19]:
resume_skills = extract_skills(clean_resume)

print("Detected Skills:")

for skill in resume_skills:
    print("✅", skill)

Detected Skills:
✅ sql


**Block 9: Job Description Analyzer + ATS Matching Score**

Input:

Resume

Python Developer with TensorFlow and Machine Learning experience

Job Description

Looking for Machine Learning Engineer.
Required skills:
Python
TensorFlow
NLP
AWS
Docker

AI Output:

ATS Score: 72%

Matched Skills:
✅ Python
✅ TensorFlow
✅ Machine Learning

Missing Skills:
❌ NLP
❌ AWS
❌ Docker

**Cell 9.1: Create Job Description Input**

For testing, create a sample job description.

In [20]:
job_description = """

We are hiring a Machine Learning Engineer.

Required Skills:

Python
Machine Learning
Deep Learning
TensorFlow
NLP
AWS
Docker
SQL

Experience with AI projects is preferred.

"""

**Cell 9.2: Clean Job Description**

In [21]:
clean_job_description = clean_text(job_description)

print(clean_job_description)

hiring machine learning engineer required skills python machine learning deep learning tensorflow nlp aws docker sql experience ai projects preferred


**Block 10: TF-IDF Vectorization**

TF-IDF converts text into numbers so ML can compare them.

In [22]:
vectorizer = TfidfVectorizer()


vectors = vectorizer.fit_transform(
    [
        clean_resume,
        clean_job_description
    ]
)


print(vectors.shape)

(2, 103)


**Cell 10.2: Calculate Similarity Score**

In [24]:
similarity_score = cosine_similarity(
    vectors[0],
    vectors[1]
)


score = similarity_score[0][0]


print("Similarity Score:", score)

Similarity Score: 0.03627890700725012


**Cell 10.3: Convert Into ATS Percentage**

In [25]:
ats_score = round(score * 100, 2)


print(
    f"ATS Resume Score: {ats_score}%"
)

ATS Resume Score: 3.63%


**Block 11: Skill Matching System**

Now we improve our ATS system.

We compare:

Resume Skills vs Job Skills

In [26]:
job_skills = extract_skills(
    clean_job_description
)


print("Required Skills:")

for skill in job_skills:
    print(skill)

Required Skills:
python
machine learning
deep learning
tensorflow
nlp
sql
aws
docker


**Cell 11.2: Find Matching Skills**

In [27]:
matched_skills = []


missing_skills = []


for skill in job_skills:

    if skill in resume_skills:
        matched_skills.append(skill)

    else:
        missing_skills.append(skill)


print("Matched Skills:")

for skill in matched_skills:
    print("✅", skill)


print("\nMissing Skills:")

for skill in missing_skills:
    print("❌", skill)

Matched Skills:
✅ sql

Missing Skills:
❌ python
❌ machine learning
❌ deep learning
❌ tensorflow
❌ nlp
❌ aws
❌ docker


**Block 12: Create Final Resume Analysis Function**

Now we combine everything.

In [28]:
def analyze_resume(resume_text, job_description):

    clean_resume_text = clean_text(resume_text)

    clean_job_text = clean_text(job_description)


    # Similarity
    vectors = vectorizer.fit_transform(
        [
            clean_resume_text,
            clean_job_text
        ]
    )


    score = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]


    ats_score = round(score*100,2)


    # Skills
    resume_skills = extract_skills(
        clean_resume_text
    )

    job_skills = extract_skills(
        clean_job_text
    )


    matched = list(
        set(resume_skills)
        &
        set(job_skills)
    )


    missing = list(
        set(job_skills)
        -
        set(resume_skills)
    )


    return {

        "ATS Score": ats_score,

        "Matched Skills": matched,

        "Missing Skills": missing

    }

**Test Our AI**

In [29]:
result = analyze_resume(
    resume_text,
    job_description
)


result

{'ATS Score': np.float64(3.63),
 'Matched Skills': ['sql'],
 'Missing Skills': ['aws',
  'tensorflow',
  'nlp',
  'docker',
  'python',
  'machine learning',
  'deep learning']}

**Block 13: Resume Improvement Suggestion Engine**

Goal:

Input:

ATS Score: 65%

Missing Skills:
AWS
Docker
NLP

AI Output:

Suggestions:

1. Add AWS experience to improve cloud skills.
2. Include NLP projects in your resume.
3. Add Docker knowledge for deployment roles.

**Cell 13.1: Create Skill Suggestion Database**

In [30]:
skill_suggestions = {

    "python":
    "Highlight Python projects and automation work.",

    "machine learning":
    "Add ML projects with datasets, models and accuracy.",

    "deep learning":
    "Include CNN, RNN or Transformer projects.",

    "tensorflow":
    "Mention TensorFlow models and deployment experience.",

    "pytorch":
    "Add PyTorch deep learning projects.",

    "nlp":
    "Add NLP projects like chatbot, sentiment analysis or text classification.",

    "computer vision":
    "Include image processing or object detection projects.",

    "aws":
    "Learn cloud deployment and mention AWS services.",

    "docker":
    "Add Docker containerization experience.",

    "sql":
    "Mention database queries and data handling experience.",

    "streamlit":
    "Add deployed AI dashboards and web applications.",

    "git":
    "Show GitHub repositories with documentation."

}


print("Suggestion Database Loaded ✅")

Suggestion Database Loaded ✅


**Cell 13.2: Generate Suggestions Function**

In [31]:
def generate_suggestions(missing_skills):

    suggestions = []

    for skill in missing_skills:

        if skill in skill_suggestions:

            suggestions.append(
                skill_suggestions[skill]
            )


    if len(suggestions) == 0:

        suggestions.append(
            "Your resume matches well. Keep improving your projects."
        )


    return suggestions

**Cell 13.3: Test Suggestion System**

In [32]:
suggestions = generate_suggestions(
    result["Missing Skills"]
)


for i, suggestion in enumerate(suggestions,1):

    print(
        f"{i}. {suggestion}"
    )

1. Learn cloud deployment and mention AWS services.
2. Mention TensorFlow models and deployment experience.
3. Add NLP projects like chatbot, sentiment analysis or text classification.
4. Add Docker containerization experience.
5. Highlight Python projects and automation work.
6. Add ML projects with datasets, models and accuracy.
7. Include CNN, RNN or Transformer projects.


**Block 14: Create Complete AI Resume Analyzer Pipeline**

Now we combine:

Resume Extraction
Cleaning
ATS Score
Skill Matching
Suggestions

In [33]:
def resume_analyzer(resume_text, job_description):

    analysis = analyze_resume(
        resume_text,
        job_description
    )


    suggestions = generate_suggestions(
        analysis["Missing Skills"]
    )


    analysis["Suggestions"] = suggestions


    return analysis

**Test Complete System**

In [34]:
final_result = resume_analyzer(
    resume_text,
    job_description
)


final_result

{'ATS Score': np.float64(3.63),
 'Matched Skills': ['sql'],
 'Missing Skills': ['aws',
  'tensorflow',
  'nlp',
  'docker',
  'python',
  'machine learning',
  'deep learning'],
 'Suggestions': ['Learn cloud deployment and mention AWS services.',
  'Mention TensorFlow models and deployment experience.',
  'Add NLP projects like chatbot, sentiment analysis or text classification.',
  'Add Docker containerization experience.',
  'Highlight Python projects and automation work.',
  'Add ML projects with datasets, models and accuracy.',
  'Include CNN, RNN or Transformer projects.']}

**Block 15: Save AI Components**

Before creating Streamlit, we save our model files.

Create folder:

In [35]:
import os

os.makedirs(
    "models",
    exist_ok=True
)

**Save Vectorizer**

In [36]:
with open(
    "models/tfidf_vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(
        vectorizer,
        file
    )


print("Vectorizer Saved ✅")

Vectorizer Saved ✅


**Save Skill Database**

In [37]:
with open(
    "models/skills.pkl",
    "wb"
) as file:

    pickle.dump(
        skills_database,
        file
    )


print("Skills Saved ✅")

Skills Saved ✅


**Download Models From Colab**

In [38]:
from google.colab import files

files.download(
    "models/tfidf_vectorizer.pkl"
)

files.download(
    "models/skills.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Block 16: Create Streamlit App**

In Google Colab create a new file:

app.py

We will write the complete Streamlit code.

**Cell 16.1: Install Streamlit**

In [39]:
!pip install streamlit

**Cell 16.2: Create app.py**

In [40]:
%%writefile app.py

import streamlit as st
import PyPDF2
import re
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# -------------------------
# Load Models
# -------------------------

with open(
    "models/skills.pkl",
    "rb"
) as file:

    skills_database = pickle.load(file)



# -------------------------
# PDF Reader
# -------------------------

def extract_pdf_text(file):

    pdf_reader = PyPDF2.PdfReader(file)

    text = ""

    for page in pdf_reader.pages:

        text += page.extract_text()

    return text



# -------------------------
# Text Cleaning
# -------------------------

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r'[^a-zA-Z0-9\s]',
        '',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text



# -------------------------
# Skill Extraction
# -------------------------

def extract_skills(text):

    found=[]

    for skill in skills_database:

        if skill in text:

            found.append(skill)


    return found



# -------------------------
# ATS Analyzer
# -------------------------

def analyze_resume(
    resume_text,
    job_text
):

    resume_clean = clean_text(
        resume_text
    )

    job_clean = clean_text(
        job_text
    )


    vectorizer = TfidfVectorizer()


    vectors = vectorizer.fit_transform(
        [
            resume_clean,
            job_clean
        ]
    )


    score = cosine_similarity(
        vectors[0],
        vectors[1]
    )[0][0]


    ats_score = round(
        score*100,
        2
    )


    resume_skills = extract_skills(
        resume_clean
    )


    job_skills = extract_skills(
        job_clean
    )


    matched = list(
        set(resume_skills)
        &
        set(job_skills)
    )


    missing = list(
        set(job_skills)
        -
        set(resume_skills)
    )


    return (
        ats_score,
        matched,
        missing
    )



# -------------------------
# Streamlit UI
# -------------------------

st.set_page_config(
    page_title="AI Resume Analyzer",
    page_icon="📄"
)


st.title(
    "📄 AI Resume Analyzer"
)


st.write(
    "Upload your resume and check ATS compatibility."
)



resume_file = st.file_uploader(
    "Upload Resume PDF",
    type=["pdf"]
)



job_description = st.text_area(
    "Paste Job Description"
)



if st.button("Analyze Resume"):


    if resume_file and job_description:


        resume_text = extract_pdf_text(
            resume_file
        )


        score, matched, missing = analyze_resume(
            resume_text,
            job_description
        )


        st.subheader(
            "ATS Score"
        )

        st.success(
            f"{score}%"
        )


        st.subheader(
            "Matched Skills"
        )


        for skill in matched:

            st.write(
                "✅",
                skill
            )



        st.subheader(
            "Missing Skills"
        )


        for skill in missing:

            st.write(
                "❌",
                skill
            )



    else:

        st.warning(
            "Please upload resume and add job description."
        )

Writing app.py


**Block 17: Test Streamlit App in Colab**

Install tunnel:

In [41]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

# Replace "YOUR_AUTH_TOKEN" with your actual ngrok authtoken
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3HHBtOZDymYKTRUHGmQYnDe5bOi_6tX6xMdPxeVUGxiqAcRai")

public_url = ngrok.connect(8501)

print(public_url)